# Joining Multiple Seasons: WTA Matches 2010–2023

We have 14 years of match data spread across separate CSV files. This notebook consolidates
them into a single player-season dataset — the input for the regression model in the next notebook.

Data: [JeffSackmann/tennis_wta](https://github.com/JeffSackmann/tennis_wta) · CC BY-NC-SA 4.0

In [ ]:
import pandas as pd
import glob

from errors import check_multiple_choice, get_notebook_logger, run_guarded

logger = get_notebook_logger()

## Stuck?

This pipeline carries several intermediate frames downstream (`df`, `player_matches`,
`season_stats`, `season_end_points`, `compiled`...). If any of them get clobbered
partway through, the cleanest fix is to **restart the kernel** and re-run everything
from the top.

In VS Code: `Kernel → Restart Kernel`, then run the cells in order again. Don't be
precious — clean state is cheap to get back, and a 14-file concat is fast.


## [Notebook Helpers]

This notebook imports helper functions from `errors.py`:
- `get_notebook_logger(...)` sets up consistent logging output.
- `check_multiple_choice(...)` validates MCQ responses with clear feedback.
- `run_guarded(...)` wraps risky steps and prints a learner-friendly error message before re-raising.

You can complete exercises without editing `errors.py`, but if an import fails, check that `errors.py` exists in the project root.

## [ASIDE] Why one file per year?

This is a common pattern in data engineering: partitioning data by time period keeps individual
files manageable and makes incremental updates straightforward (add a new year, don't rewrite
everything). The `glob` module lets us discover files by pattern so we're not hardcoding filenames.

## Exercise 1: Find the Match Files

**Which glob pattern finds exactly the WTA match files (excluding qualifying/ITF files)?**

- A) `"../data/*.csv"` — matches every CSV in `data/`, including non-match files
- B) `"../data/wta_matches_*.csv"` — `*` is greedy, so this matches qualifying/ITF files too (e.g. `wta_matches_qual_itf_2023.csv`)
- C) `"../data/wta_matches_[0-9]*.csv"` — `[0-9]` is a character class, so this requires a digit immediately after `wta_matches_`
- D) `"../data/**/*.csv"` — recurses into subdirectories, overkill here

We want only the year-numbered files (e.g. `wta_matches_2010.csv` through `wta_matches_2023.csv`).
Set `answer` below.


In [ ]:
answer = "C"

options = {
    "A": "../data/*.csv",
    "B": "../data/wta_matches_*.csv",
    "C": "../data/wta_matches_[0-9]*.csv",
    "D": "../data/**/*.csv",
}

check_multiple_choice(
    answer=answer,
    options=options,
    is_correct=lambda selected, choices: "[0-9]*" in choices[selected],
    exercise_label="JOIN_DATA Exercise 1",
    incorrect_message="Close, but choose the pattern that targets year-numbered files only.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 1 completed.")

In [ ]:
pattern = "../data/wta_matches_[0-9]*.csv"

# Assign the function from the `glob` module that expands a pattern into a list of paths.
glob_fn = glob.glob

all_files = run_guarded(
    step_label="JOIN_DATA Exercise 2",
    action=lambda: glob_fn(pattern),
    user_error_message="Could not discover files. Check that glob_fn is callable and pattern is valid.",
    logger=logger,
)

files = [f for f in all_files if any(str(yr) in f for yr in range(2010, 2024))]
files = sorted(files)
logger.info("Discovered %d files for 2010-2023.", len(files))
print(f"Found {len(files)} files")
print(files[:3], "...")
logger.info("JOIN_DATA Exercise 2 file discovery completed.")


## [ASIDE] Loops vs copy-paste

If you've ever applied the same transformation across 14 Excel sheets manually, a loop is Python's
answer. Same logic, none of the repetition. The list we're building (`dfs`) collects all the
DataFrames before we concatenate — more efficient than concatenating inside the loop.

## [ASIDE] `ignore_index=True`

When you stack DataFrames vertically, each one's original row index comes with it — so the
combined frame ends up with duplicate index values (multiple rows labelled 0, 1, 2...).
Passing `ignore_index=True` resets the result to a clean sequential index. Usually what you
want after a concat.


## Exercise 2: Load and Concatenate

You have a list of file paths in `files`. The plan is:

1. Iterate the paths, reading each into a DataFrame (use `pd.read_csv(filepath, low_memory=False)` —
   `low_memory=False` tells pandas to read the whole file before inferring column dtypes,
   which avoids mixed-type warnings).
2. Collect each loaded DataFrame in the `dfs` list.
3. Combine the list into one DataFrame using a single pandas function (see the aside above on
   `ignore_index=True` for the second argument).


In [ ]:
dfs = []
for filepath in files:
    year_df = pd.read_csv(filepath, low_memory=False)
    dfs.append(year_df)

# Step 4: choose a concat callable and ignore_index setting, then combine dfs.
concat_fn = pd.concat
use_ignore_index = True
df = run_guarded(
    step_label="JOIN_DATA Exercise 2",
    action=lambda: concat_fn(dfs, ignore_index=use_ignore_index),
    user_error_message="Could not concatenate file DataFrames.",
    logger=logger,
)
print(f"Combined shape: {df.shape}")
logger.info("JOIN_DATA Exercise 2 concatenation completed with shape %s.", df.shape)

### If This Fails, Check

- `glob_fn` and `concat_fn` are callables.
- You discovered files before concatenating (the `files` list is not empty).
- You ran cells in order so `files` and `dfs` exist.
- The `data/` folder is present in the project root.

## Exercise 3: Inspect the Combined Data

In [ ]:
def _exercise_3_inspect():
    shape_value = df.shape
    print(f"Shape: {shape_value}")

    missing = df.isnull().sum().sort_values(ascending=False)
    print(f"\nTop 10 columns by missingness:\n{missing.head(10)}")

run_guarded(
    step_label="JOIN_DATA Exercise 3",
    action=_exercise_3_inspect,
    user_error_message="Could not inspect the combined DataFrame.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 3 completed.")

## Exercise 4: Filter and Select

Two steps. First, drop rows where any of the key stat columns are missing — pandas has a
DataFrame method that takes a `subset=` argument for exactly this. Second, narrow `df` down
to just the columns we need for feature engineering, and call `.copy()` on the result so
later assignments don't trip a `SettingWithCopyWarning`.


In [ ]:
STAT_COLS = ["w_ace", "w_svpt", "w_1stIn", "w_df", "l_ace", "l_svpt", "l_1stIn", "l_df"]
KEEP_COLS = [
    "tourney_date", "surface",
    "winner_name", "winner_rank", "winner_rank_points",
    "w_ace", "w_svpt", "w_1stIn", "w_df",
    "loser_name", "loser_rank", "loser_rank_points",
    "l_ace", "l_svpt", "l_1stIn", "l_df",
]
# Note: 'surface' is retained here but not used in the player-level reshape below.
# It could be useful for surface-level analysis if you want to extend this pipeline later.

# Drop rows from `df` where any of STAT_COLS are NaN.
df = run_guarded(
    step_label="JOIN_DATA Exercise 4 (drop NaNs)",
    action=lambda: df.dropna(subset=STAT_COLS),
    user_error_message="Could not drop rows with null stat columns. Check the lambda body.",
    logger=logger,
)

# Narrow `df` to KEEP_COLS only and return an independent copy.
df = run_guarded(
    step_label="JOIN_DATA Exercise 4 (select cols)",
    action=lambda: df[KEEP_COLS].copy(),
    user_error_message="Could not select the required columns. Check the lambda body.",
    logger=logger,
)
print(f"Shape after filtering: {df.shape}")
logger.info("JOIN_DATA Exercise 4 completed with shape %s.", df.shape)


## [ASIDE] Feature selection early

Carrying unnecessary columns through an analysis slows things down and makes DataFrames
harder to read. Selecting only what you need at the point of loading is a habit worth
building — the equivalent of writing a targeted SQL SELECT rather than SELECT *.

## Exercise 5: Build a Player-Centric View

The match data has one row per match with winner-prefixed and loser-prefixed columns.
To aggregate by player, we need to reshape: create one row per player appearance
(win or loss) with consistent column names.

In [ ]:
def build_player_matches(df: pd.DataFrame) -> pd.DataFrame:
    """
    Reshape match data into one row per player per match.

    Each match contributes two rows: one for the winner, one for the loser.
    Stat columns are renamed to be player-centric (ace, svpt, first_in, df_count).
    A 'won' column (1/0) records the match outcome.

    Parameters
    ----------
    df : pd.DataFrame
        Raw match DataFrame with winner_*/loser_* columns.

    Returns
    -------
    pd.DataFrame
        One row per player per match with columns:
        tourney_date, player, rank, rank_points, ace, svpt, first_in, df_count, won, year.
    """
    # Step 1: Build a DataFrame of winner-side rows. Select tourney_date, winner_name,
    #         winner_rank, winner_rank_points, and the w_* stat columns. Use .copy() so
    #         later renames don't mutate the input.
    winner_rows = df[[
        "tourney_date", "winner_name", "winner_rank", "winner_rank_points",
        "w_ace", "w_svpt", "w_1stIn", "w_df"
    ]].copy()

    # Step 2: Rename the winner-side columns to the player-centric schema.
    winner_rows.columns = [
        "tourney_date", "player", "rank", "rank_points",
        "ace", "svpt", "first_in", "df_count"
    ]

    # Step 3: Mark these rows as wins.
    winner_rows["won"] = 1

    # Step 4: Repeat Steps 1–3 for the loser side. Same target schema, but `won = 0`.
    loser_rows = df[[
        "tourney_date", "loser_name", "loser_rank", "loser_rank_points",
        "l_ace", "l_svpt", "l_1stIn", "l_df"
    ]].copy()
    loser_rows.columns = [
        "tourney_date", "player", "rank", "rank_points",
        "ace", "svpt", "first_in", "df_count"
    ]
    loser_rows["won"] = 0

    # Step 5: Combine the winner and loser frames into one.
    player_matches = pd.concat([winner_rows, loser_rows], ignore_index=True)

    # Step 6: Derive a `year` column from tourney_date (integer like 20230115).
    player_matches["year"] = (
        pd.to_datetime(player_matches["tourney_date"], format="%Y%m%d").dt.year
    )

    # Step 7: Return the combined player-match DataFrame.
    return player_matches


player_matches = run_guarded(
    step_label="JOIN_DATA Exercise 5",
    action=lambda: build_player_matches(df),
    user_error_message="Could not build player-level match rows.",
    logger=logger,
)
print(f"Player-match rows: {player_matches.shape}")
player_matches.head()
logger.info("JOIN_DATA Exercise 5 completed with shape %s.", player_matches.shape)


## Exercise 6: Compute Per-Match Rates

Add derived rate columns before aggregating.

In [ ]:
first_in_col = "first_in"
svpt_col = "svpt"
ace_col = "ace"
df_col = "df_count"

def _exercise_6_rates():
    player_matches["first_serve_pct"] = player_matches[first_in_col] / player_matches[svpt_col]
    player_matches["ace_rate"] = player_matches[ace_col] / player_matches[svpt_col]
    player_matches["df_rate"] = player_matches[df_col] / player_matches[svpt_col]

run_guarded(
    step_label="JOIN_DATA Exercise 6",
    action=_exercise_6_rates,
    user_error_message="Could not compute per-match rates.",
    logger=logger,
)
logger.info("JOIN_DATA Exercise 6 completed.")

## [ASIDE] Aggregating with `groupby`

`df.groupby(keys).agg(...)` is the pandas equivalent of SQL `GROUP BY`. With **named
aggregations** you spell out each output column in the call:

```python
df.groupby(["player", "year"]).agg(
    win_count=("won", "sum"),
    total_matches=("won", "count"),
    avg_rank=("rank", "mean"),
)
```

Each keyword becomes a new column whose name is your choice; the tuple is
`(source_column, agg_func)`. `agg_func` can be a string like `"sum"`, `"count"`, `"mean"`,
`"max"`, etc. Chain `.reset_index()` at the end to flatten the group keys back into regular
columns instead of leaving them in the index.

## Exercise 7: Aggregate to Player-Season Level

Use `groupby` to collapse the per-match rows into one row per `(player, year)`. You need a
sum of `won` (wins), a count of `won` (total matches), mean of the rate columns and rank,
then a derived `win_rate = win_count / total_matches`.


In [ ]:
def aggregate_player_season(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate per-match player data to one row per player per season.

    Parameters
    ----------
    player_matches : pd.DataFrame
        Output of build_player_matches() with rate columns added.

    Returns
    -------
    pd.DataFrame
        Columns: player, year, win_count, total_matches, win_rate,
                 avg_first_serve_pct, avg_ace_rate, avg_df_rate, avg_rank.
    """
    agg = player_matches.groupby(["player", "year"]).agg(
        win_count=("won", "sum"),
        total_matches=("won", "count"),
        avg_first_serve_pct=("first_serve_pct", "mean"),
        avg_ace_rate=("ace_rate", "mean"),
        avg_df_rate=("df_rate", "mean"),
        avg_rank=("rank", "mean"),
    ).reset_index()
    agg["win_rate"] = agg["win_count"] / agg["total_matches"]
    return agg


season_stats = run_guarded(
    step_label="JOIN_DATA Exercise 7",
    action=lambda: aggregate_player_season(player_matches),
    user_error_message="Could not aggregate player-season statistics.",
    logger=logger,
)
print(f"Player-season rows: {season_stats.shape}")
season_stats.head()
logger.info("JOIN_DATA Exercise 7 completed with shape %s.", season_stats.shape)

## Exercise 8: Season-End Ranking Points

For each player-season, take the ranking points from their last match of the year. Not
official year-end ranking, but a good approximation.

### Pattern: "last row per group"

To pick a single row per group you generally:

1. Sort the whole frame by the column that defines "first/last" (here, `tourney_date`).
2. Group by the keys (`player`, `year`).
3. Use `.nth(-1)` to take the final row in each group (negative indexing works as you'd
   expect; `.nth(0)` would give the first).
4. Reset the index and select the columns you want.

Finally, rename `rank_points` → `ranking_points` so the column reads naturally downstream.


In [ ]:
def get_season_end_points(player_matches: pd.DataFrame) -> pd.DataFrame:
    """
    Extract a proxy season-end ranking points for each player-year.

    Takes the ranking points recorded in the player's final match of each season.
    Note: this is a proxy, not official year-end ranking points.

    Parameters
    ----------
    player_matches : pd.DataFrame
        Output of build_player_matches().

    Returns
    -------
    pd.DataFrame
        Columns: player, year, ranking_points.
    """
    sorted_matches = player_matches.sort_values("tourney_date")
    last_match = (
        sorted_matches.groupby(["player", "year"])
        .nth(-1)
        .reset_index()[["player", "year", "rank_points"]]
    )
    last_match = last_match.rename(columns={"rank_points": "ranking_points"})
    return last_match


season_end_points = run_guarded(
    step_label="JOIN_DATA Exercise 8",
    action=lambda: get_season_end_points(player_matches),
    user_error_message="Could not compute season-end ranking points.",
    logger=logger,
)
season_end_points.head()
logger.info("JOIN_DATA Exercise 8 completed with shape %s.", season_end_points.shape)

## [ASIDE] Is this proxy biased?

We're using `rank_points` from the player's *last match of the year* as a stand-in for the
official year-end ranking. That's pragmatic, but it carries a subtle distortion.

Consider a top-100 player who plays a deep run at the WTA Finals in November — their last
match's `rank_points` reflects the ranking *before* those Finals points are awarded. A
journeywoman whose last match was a Round-of-32 loss at a 250-tier event in October has a
much more "settled" final-match snapshot.

Net effect: the proxy probably *understates* the season-end points of top players (whose
last matches tend to be at high-stakes, late-season events) more than it does for the rest
of the tour. That's a systematic bias to hold loosely when you read the regression
coefficients.


## Exercise 9: Merge and Write Output

DataFrames have a `.merge()` method — the pandas analogue of SQL `JOIN`. The signature is
`left.merge(right, on=<key columns>, how=<"inner"|"left"|"right"|"outer">)`. `on` accepts a
single column name or a list of names that must exist on both sides.

We want to attach each `(player, year)`'s season-end ranking points onto its season stats —
keeping only rows present in **both** frames.


In [ ]:
merge_fn = season_stats.merge
merge_keys = ["player", "year"]
merge_how = "inner"

compiled = run_guarded(
    step_label="JOIN_DATA Exercise 9",
    action=lambda: merge_fn(season_end_points, on=merge_keys, how=merge_how),
    user_error_message="Could not merge season_stats with season_end_points.",
    logger=logger,
)
compiled = compiled.dropna()
print(f"Final compiled shape: {compiled.shape}")
compiled.head()
logger.info("JOIN_DATA Exercise 9 merge completed with shape %s.", compiled.shape)

In [ ]:
import os; os.makedirs("../output", exist_ok=True)

output_path = "../output/wta_compiled.csv"
write_index = False
run_guarded(
    step_label="JOIN_DATA Exercise 9 (Write Output)",
    action=lambda: compiled.to_csv(output_path, index=write_index),
    user_error_message="Could not write compiled output.",
    logger=logger,
)

print(f"Written to {output_path}")
logger.info("JOIN_DATA Exercise 9 write completed: %s", output_path)

## Git Signpost

Commit the notebook and note that the compiled CSV is a generated artefact (it lives in
`output/` which is gitignored — the learner regenerates it by running this notebook).

```bash
git add JOIN_DATA.ipynb
git commit -m "complete JOIN_DATA notebook"
```